In [1]:
# # remember to resteart kernell
# ! pip install -e ../../wazeasy
# ! pip install pandas "dask[complete]" pyarrow plotly
# ! pip install --upgrade s3fs
# ! pip install geopandas
# ! pip install seaborn
# ! pip install folium matplotlib mapclassifys
# ! pip install h3
# ! pip install dask-geopandas
# ! pip install folium matplotlib mapclassify
# ! pip install altair altair_tiles
# ! pip install "attaviz @ git+https://github.com/datapartnership/attaviz.git"

In [1]:
import pandas as pd
import geopandas as gpd
import dask.dataframe as dd
import dask_geopandas
from dask import delayed, compute
from wazeasy import utils, plots, reports
import geopandas as gpd

import yaml
import altair as alt
alt.renderers.enable('mimetype')
import re

# Processing the data into TCI tiles

In [2]:
cities = {
    # 'new_york': {'path':'s3://wbg-waze/bq/US/811newyork/jams/*parquet',
    #              'projected_crs':'EPSG:32618',
    #              'time_zone':'America/New_York',
    #              'gdf':'../data/01_grid_nyc_1km.gpkg'
    #             },
    'buenos_aires': {'path':'s3://wbg-waze/bq/AR/811buenosaires/jams/*parquet',
                     'projected_crs':'EPSG:32721',
                     'time_zone':'America/Buenos_Aires',
                     'gdf':'../data/02_grid_buenos_aires_1km_osm.gpkg'
                    },
    'baghdad': {'path':'s3://wbg-waze/bq/IQ/811baghdad/jams/*parquet',
                 'projected_crs':'EPSG:32638',
                 'time_zone':'Asia/Baghdad',
                 'gdf':'../data/03_grid_baghdad_governorate_1km_osm.gpkg'
                },
    'mexico': {'path':'s3://wbg-waze/bq/MX/811mexico/jams/*parquet',
                 'projected_crs':'EPSG:32614',
                 'time_zone':'America/Mexico_City',
                 'gdf':'../data/04_grid_greater_mexico_city_1km_osm.gpkg'
                }
}
storage_options = {'profile': 'ddp'}

In [ ]:
for city, city_data in cities.items():
    print(city)
    path = city_data['path']
    projected_crs = city_data['projected_crs']
    time_zone = city_data['time_zone']
    gdf = city_data['gdf']
    ddf = utils.load_data(path, 
                      storage_options = storage_options, file_type = 'parquet', 
                      filter_level_5 = True, 
                      usecols = ['ts', 'geoWKT', 'uuid', 'level','length'])
    utils.handle_time(ddf, time_zone)
    tiles = gpd.read_file(gdf)
    result_overlay = utils.split_jams_into_geometries(ddf, tiles, projected_crs)
    daily_tci = result_overlay.groupby(['date', 'spatial_id'])[['length']].sum().compute()
    print(f'original {ddf.length.sum().compute()} and processed {daily_tci.length.sum()}')
    daily_tci.to_csv(f'./results/tci_2025_{city}.csv')

buenos_aires
original 21810181687 and processed 21804059273.450092
baghdad
original 89313395517 and processed 83847268652.8318
mexico
original 152792720153 and processed 68637833157.2398
